# Hyperparameter Tunnig

This script is using the data pipeline to clean the data.
It will use Hyperopt for hyperparameter tuning and safe the best model via mlflow.

The models will be tested against:
- 1 day
- 1 week
- 2 weeks
- 4 weeks
- 1 quarter
- 2 quarters
- 3 quarters
- 4 quarters

As well as based on data need, this will be evaluated based on CV.

The models to be tuned are:
- SARIMAX
- Tripple Exponential Smoothing
- Prophet
- XG Boost
- Linear Regression
- Random Forest
- LSTM
- Temporal Fusion Transformer (TFT)
- Deep Autoregression Models

# Libraries

In [25]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import sys
import os
from darts import TimeSeries
from darts.models import Prophet, ARIMA, ExponentialSmoothing
from darts.utils.utils import ModelMode, SeasonalityMode
from darts.metrics import mae, mape, rmse
from hyperopt import hp
import mlflow

# Add the project root to the python path
sys.path.append(os.path.abspath(".."))
from src.processing import DateFeatureTransformer, TimeSeriesWrangler, LagFeatureTransformer, WindowFeatureTransformer
from src.evaluation import DartsObjective, TimeSeriesOptimizer


In [26]:
mlflow.set_tracking_uri("file:../mlruns")
#mlflow.set_tracking_uri("sqlite:../ipynb/mlflow.db")

# Loading Data

In [27]:
# define path
path = "../data/raw/"

In [28]:
# oil data
oil_df = pd.read_csv(path + "oil.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='dcoilwtico', 
    freq='D', 
    fill_method='ffill'
)

# Run the cleaning logic
oil = wrangler.clean(oil_df)

In [29]:
# timeseries data
timeseries_df = pd.read_csv(path + "timeseries.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='unit_sales', 
    freq='D', 
    fill_method='zeros'
)

# Run the cleaning logic
timeseries = wrangler.clean(timeseries_df)

# Variables

In [30]:
# Defining constants
random_seed = 42
# Change these if you df has different column names
target_col = 'unit_sales'
time_col = 'date'
forecast_horizon = 7

# SARIMAX

In [31]:
# Now, when you scale, the index stays untouched
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

# 1. Initialize the scikit-learn scaler you want
base_scaler = StandardScaler()

# 2. Wrap it in the Darts Scaler
transformer = Scaler(scaler=base_scaler)

In [32]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Suppress the warning so it doesn't flood your console
warnings.simplefilter('ignore', ConvergenceWarning)

In [33]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_holiday', 'is_payday'], payday_val=15, country='EC', drop_date_col=False))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# Set the date as the index first
df = timeseries_oil.set_index(time_col)

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()
series = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=target_col, freq='D')
exog = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=all_exog_features, freq='D')
exog = transformer.fit_transform(exog)

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
    'SARIMAX': {
        'class': ARIMA,
        'space': {
            # 1. Feature Selection: Toggles each feature on or off
            'selected_features': [hp.choice(f'feat_{f}', [None, f]) for f in all_exog_features],
            
            # 2. Standard SARIMAX Params
            'p': hp.quniform('p', 4, 5, 1),
            'd': 1, #hp.choice('d', [0, 1]),
            'q': hp.quniform('q', 1, 2, 1),
            
            # 3. Seasonal Params (Weekly Seasonality for Ecuador Sales)
            'seasonal_order': (
            hp.quniform('P', 1, 2, 1),
            hp.choice('D', [0, 1]),
            hp.quniform('Q', 1, 2, 1),
            7)#,
            #'trend': hp.choice('trend', ['n', 't'])
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name="SARIMAX_"+str(forecast_horizon))

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=forecast_horizon,       # 7-day forecast
        metric=mae,                     # Optimization metric
        exog=exog,                      # Your Darts exogenous TimeSeries
        max_evals=20                    # Run 50 trials per model
    )



                     date  unit_sales  date_is_weekend  date_is_holiday  \
date             1.000000   -0.010188         0.004833         0.016577   
unit_sales      -0.010188    1.000000         0.685608         0.008411   
date_is_weekend  0.004833    0.685608         1.000000        -0.021108   
date_is_holiday  0.016577    0.008411        -0.021108         1.000000   
date_is_payday  -0.002440   -0.013588         0.013869        -0.044849   
dcoilwtico       0.340182    0.003497         0.006537         0.008804   

                 date_is_payday  dcoilwtico  
date                  -0.002440    0.340182  
unit_sales            -0.013588    0.003497  
date_is_weekend        0.013869    0.006537  
date_is_holiday       -0.044849    0.008804  
date_is_payday         1.000000   -0.008234  
dcoilwtico            -0.008234    1.000000  
Resuming SARIMAX: Found 20 previous trials.
Optimization for SARIMAX already completed 20 evals. Skipping search.
Logging Champion SARIMAX to MLflow...


# Tripple Exponential Smooting

In [34]:
series = TimeSeries.from_dataframe(timeseries, time_col=time_col, value_cols=target_col, freq='D')

# Define your models and search spaces
registry = {
    'Triple Exponential Smoothing': {
        'class': ExponentialSmoothing,
        'space': {
            'trend': hp.choice('trend', [ModelMode.ADDITIVE, ModelMode.MULTIPLICATIVE]),
            'seasonal': hp.choice('seasonal', [SeasonalityMode.ADDITIVE, SeasonalityMode.MULTIPLICATIVE]),
            'damped': hp.choice('damped', [True, False]),
            'seasonal_periods': 7
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name="Triple Exponential Smoothing_"+str(forecast_horizon))

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=forecast_horizon,       # 7-day forecast
        metric=mae,                     # Optimization metric
        exog=None,                      # No exogenous variables for ETS
        max_evals=20                    # Run 50 trials per model
    )



Resuming Triple Exponential Smoothing: Found 20 previous trials.
Optimization for Triple Exponential Smoothing already completed 20 evals. Skipping search.
Logging Champion Triple Exponential Smoothing to MLflow...


# Prophet

In [35]:
# Now, when you scale, the index stays untouched
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

# 1. Initialize the scikit-learn scaler you want
base_scaler = StandardScaler()

# 2. Wrap it in the Darts Scaler
transformer = Scaler(scaler=base_scaler)

In [36]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_payday'], payday_val=15, country='EC', drop_date_col=False))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# Set the date as the index first
df = timeseries_oil.set_index(time_col)

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()
series = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=target_col, freq='D')
exog = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=all_exog_features, freq='D')
exog = transformer.fit_transform(exog)

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
    'Prophet': {
        'class': Prophet,
        'space': {
            # 1. Feature Selection: Toggles each feature on or off
            'selected_features': [hp.choice(f'feat_{f}', [None, f]) for f in all_exog_features],
            
            # 2. Standard Prophet Params
            'changepoint_prior_scale': hp.loguniform('changepoint_prior_scale', np.log(0.001), np.log(0.5)),
            'seasonality_prior_scale': hp.loguniform('seasonality_prior_scale', np.log(0.01), np.log(10.0)),
            'holidays_prior_scale': hp.loguniform('holidays_prior_scale', np.log(0.01), np.log(10.0)),
            'seasonality_mode': hp.choice('seasonality_mode', ['additive', 'multiplicative']),
            'changepoint_range': hp.uniform('changepoint_range', 0.8, 0.95)
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name="Prophet_"+str(forecast_horizon))

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=forecast_horizon,       # 7-day forecast
        metric=mae,                     # Optimization metric
        exog=exog,                      # Your Darts exogenous TimeSeries
        max_evals=10                    # Run 50 trials per model
    )



                     date  unit_sales  date_is_weekend  date_is_payday  \
date             1.000000   -0.010188         0.004833       -0.002440   
unit_sales      -0.010188    1.000000         0.685608       -0.013588   
date_is_weekend  0.004833    0.685608         1.000000        0.013869   
date_is_payday  -0.002440   -0.013588         0.013869        1.000000   
dcoilwtico       0.340182    0.003497         0.006537       -0.008234   

                 dcoilwtico  
date               0.340182  
unit_sales         0.003497  
date_is_weekend    0.006537  
date_is_payday    -0.008234  
dcoilwtico         1.000000  
Resuming Prophet: Found 5 previous trials.
Running Hyperopt for Prophet up to 10 evals...
 50%|█████     | 5/10 [00:00<?, ?trial/s, best loss=?]

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:13 - cmdstanpy - INFO - Chain [1] start processing

22:14:13 - cmdstanpy - INFO - Chain [1] done processing

22:14:14 - cmdstanpy - INFO - Chain [1] start processing

22:14:14 - cmdstanpy - INFO - Chain [1] done processing

22:14:14 - cmdstanpy - INFO - Chain [1] start processing

22:14:14 - cmdstanpy -

 60%|██████    | 6/10 [00:09<00:38,  9.54s/trial, best loss: 97.53315677724787]

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy - INFO - Chain [1] done processing

22:14:23 - cmdstanpy - INFO - Chain [1] start processing

22:14:23 - cmdstanpy -

 70%|███████   | 7/10 [00:18<00:27,  9.10s/trial, best loss: 97.53315677724787]

...

 90%|█████████ | 9/10 [00:42<00:10, 10.79s/trial, best loss: 97.53315677724787]

22:14:55 - cmdstanpy - INFO - Chain [1] start processing

22:14:55 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy - INFO - Chain [1] done processing

22:14:56 - cmdstanpy - INFO - Chain [1] start processing

22:14:56 - cmdstanpy -

100%|██████████| 10/10 [00:51<00:00, 10.29s/trial, best loss: 97.53315677724787]

22:15:05 - cmdstanpy - INFO - Chain [1] start processing

22:15:05 - cmdstanpy - INFO - Chain [1] done processing




Logging Champion Prophet to MLflow...


# XG Boost

# Linear Regression

# Random Forest

# LSTM

# TFT

# Deep Autoregression Models